# PrivateLocalAgent

**Kernel → Restart Kernel**，然后只运行下面 **一个** 代码单元格。  
会加载真实智能体 → 启动网页 → `rc-tunnel` 公网 → 本页嵌入完整 UI。

In [ ]:
import os, sys, subprocess, re, shutil
from pathlib import Path
from IPython.display import display, HTML, clear_output

def log(msg): print(msg, flush=True)

ROOT = Path("/workspace/Radeon-hackathon-2026-07")
if not (ROOT / "src" / "config.py").is_file():
    here = Path.cwd()
    ROOT = here.parent if here.name == "notebooks" else here
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
log(f"ROOT={ROOT}")

persist = Path("/workspace/persistence")
if not persist.is_dir():
    persist = Path("/persistent")
if persist.is_dir():
    os.environ.setdefault("PLA_DATA_ROOT", str(persist / "PrivateLocalAgent"))
    os.environ.setdefault("HF_HOME", str(persist / "huggingface"))
    Path(os.environ["PLA_DATA_ROOT"]).mkdir(parents=True, exist_ok=True)
    Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("USE_ROCM_AITER_ROPE_BACKEND", "0")
os.environ["PLA_NOTEBOOK_UI_PORT"] = "7900"
os.environ["PLA_ALLOW_PUBLIC"] = "1"
os.environ["HTTP_HOST"] = "127.0.0.1"
os.environ["HTTP_PORT"] = "7900"
os.environ["PLA_OPEN_BROWSER"] = "0"
os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ.get("PATH", "")

log("[0] deps...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "chromadb", "sentence-transformers", "pypdf", "pyyaml", "python-dotenv",
    "pydantic", "openai", "transformers", "accelerate", "safetensors",
    "sentencepiece", "Pillow", "rapidocr-onnxruntime",
])

from src.agent.agent import PrivateAgent
from src.agent.multi_agent import MultiAgentOrchestrator
from src.agent.tools import ToolRegistry
from src.apps.judge_script import ensure_judge_ocr_image
from src.config import load_settings
from src.llm.backend import build_llm
from src.memory.memory import SessionMemory
from src.privacy.audit import AuditTrail
from src.rag.store import VectorStore
from src.skills import SkillRegistry
from src.app.notebook_visual import launch_notebook_visual

settings = load_settings()
upload_dir = settings.resolve(settings.paths.upload_dir)
ensure_judge_ocr_image(upload_dir)

log("[1] knowledge base...")
store = VectorStore(settings)
store.ensure_sample_docs(settings.resolve(settings.paths.sample_docs))
log(f"    chunks={store.count()}")

memory = SessionMemory(settings.resolve(settings.agent.memory_path))
skills = SkillRegistry(settings.resolve(settings.paths.generated_projects))
tools = ToolRegistry(store, memory, upload_dir, skill_registry=skills)
audit = AuditTrail(settings.resolve("data/memory/audit.jsonl"))

log("[2] load local LLM on Radeon/ROCm (real agent)...")
llm = build_llm(settings.llm)
agent = PrivateAgent(llm, tools, memory, settings.agent.max_steps, audit=audit)
orch = MultiAgentOrchestrator(agent, tools)
log("ready — launching UI (web + rc-tunnel)")

# One-shot: attach orch → local web → rc-tunnel → iframe + external link
ui = launch_notebook_visual(orch, default_mode="chat")
log(f"local={ui.local_url}")
log(f"public={ui.public_url}")
log("用页面里的上传/模式/发送测试智能体；外部链接同步可用。")